# Olist E-Commerce: Growth Diagnostic

**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)  
**Period:** 2016 to 2018  
**Orders:** 99,441 total | 96,478 delivered  
**Tools:** Python, pandas, matplotlib  

---

### The Question
Olist has strong order volumes. Why is revenue not compounding?  
What is holding growth back?

### Structure of This Notebook
1. Load and join all 8 tables
2. Repeat purchase rate analysis
3. Delivery performance vs customer satisfaction
4. Seller quality and revenue at risk
5. Geographic revenue and satisfaction breakdown
6. Payment behaviour analysis
7. RFM customer segmentation

## 0. Setup and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Chart styling
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.grid': True,
    'grid.alpha': 0.2,
    'grid.linestyle': '--',
    'axes.axisbelow': True,
})

NAVY  = '#1F3864'
BLUE  = '#185FA5'
RED   = '#C00000'
GREEN = '#107C10'
GOLD  = '#E8A800'
GRAY  = '#555555'
TEAL  = '#0F6E56'

In [ ]:
# Load all 8 tables
# Update the path to wherever you saved the Kaggle download
PATH = 'data/'

orders    = pd.read_csv(PATH + 'olist_orders_dataset.csv')
customers = pd.read_csv(PATH + 'olist_customers_dataset.csv')
items     = pd.read_csv(PATH + 'olist_order_items_dataset.csv')
payments  = pd.read_csv(PATH + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(PATH + 'olist_order_reviews_dataset.csv')
products  = pd.read_csv(PATH + 'olist_products_dataset.csv')
sellers   = pd.read_csv(PATH + 'olist_sellers_dataset.csv')

# Parse all date columns
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'order_approved_at'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print('Tables loaded:')
for name, df in [('orders',orders), ('customers',customers), ('items',items),
                 ('payments',payments), ('reviews',reviews),
                 ('products',products), ('sellers',sellers)]:
    print(f'  {name:20s} {df.shape[0]:,} rows')

In [ ]:
# Build master table - join all tables once, use throughout
# Filter to delivered orders only for all revenue analysis

delivered = orders[orders['order_status'] == 'delivered'].copy()

master = (
    delivered
    .merge(customers[['customer_id','customer_unique_id','customer_state']],
           on='customer_id', how='left')
    .merge(items[['order_id','product_id','seller_id','price','freight_value']],
           on='order_id', how='left')
    .merge(products[['product_id','product_category_name']],
           on='product_id', how='left')
    .merge(reviews[['order_id','review_score']].drop_duplicates('order_id'),
           on='order_id', how='left')
    .merge(sellers[['seller_id','seller_state']],
           on='seller_id', how='left')
)

master['order_value']  = master['price'] + master['freight_value']
master['month']        = master['order_purchase_timestamp'].dt.to_period('M')
master['days_late']    = (
    master['order_delivered_customer_date']
    - master['order_estimated_delivery_date']
).dt.days

print(f'Master table: {len(master):,} rows | {len(master.columns)} columns')
print(f'Delivered orders: {delivered["order_id"].nunique():,}')
print(f'Unique customers: {master["customer_unique_id"].nunique():,}')
print(f'Total revenue: R${master["order_value"].sum():,.0f}')

## 1. Revenue Trend

In [ ]:
rev = (
    master
    .groupby('month')
    .agg(revenue=('order_value','sum'), orders=('order_id','nunique'))
    .reset_index()
)
# Focus on full months 2017 onwards
rev = rev[(rev['month'] >= '2017-01') & (rev['month'] <= '2018-08')]
rev['month_str'] = rev['month'].astype(str)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.06})

ax1.fill_between(range(len(rev)), rev['revenue'], alpha=0.12, color=NAVY)
ax1.plot(range(len(rev)), rev['revenue'], color=NAVY, linewidth=2.2,
         marker='o', markersize=5, markerfacecolor='white',
         markeredgecolor=NAVY, markeredgewidth=1.8)
ax1.set_ylabel('Monthly Revenue (R$)', color=GRAY, fontsize=10)
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M' if x >= 1e6 else f'R${x/1e3:.0f}K')
)
ax1.set_title('Revenue and Order Volume Trend (2017 to 2018)',
              fontsize=12, color='#1A1A1A', fontweight='bold', pad=14, loc='left')

ax2.bar(range(len(rev)), rev['orders'], color=BLUE, alpha=0.75,
        width=0.65, edgecolor='white')
ax2.set_ylabel('Orders', color=GRAY, fontsize=9)
ax2.set_xticks(range(len(rev)))
ax2.set_xticklabels(rev['month_str'], rotation=30, ha='right', fontsize=8.5)
ax2.spines['bottom'].set_color('#CCCCCC')

plt.tight_layout()
plt.savefig('reports/chart_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Peak month revenue: R${rev["revenue"].max():,.0f}')
print(f'Peak month orders:  {rev["orders"].max():,}')

## 2. Repeat Purchase Rate

**Note on customer_id vs customer_unique_id:**  
The customers table has two ID columns. `customer_id` is generated per order, 
meaning one real customer who places 3 orders gets 3 different customer_ids.  
Using `customer_id` to count customers inflates the total and makes every 
customer look like a first-time buyer.  
All analysis below uses `customer_unique_id` which is one ID per real person.

In [ ]:
# Count orders per unique customer
cust_orders = (
    delivered
    .merge(customers[['customer_id','customer_unique_id']], on='customer_id')
    .groupby('customer_unique_id')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'order_count'})
)

total_cust  = len(cust_orders)
repeat_cust = len(cust_orders[cust_orders['order_count'] > 1])
repeat_rate = repeat_cust / total_cust * 100

print(f'Total unique customers : {total_cust:,}')
print(f'Repeat customers       : {repeat_cust:,}')
print(f'Repeat rate            : {repeat_rate:.1f}%')
print()
print('Order frequency breakdown:')
print(cust_orders['order_count'].value_counts().head(6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Donut chart
ax = axes[0]
one_time = total_cust - repeat_cust
ax.pie(
    [one_time, repeat_cust],
    colors=['#E8ECF0', NAVY],
    explode=(0, 0.06),
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    autopct='%1.1f%%',
    pctdistance=0.72
)
ax.set_title('Customer Repeat Rate', fontsize=11, fontweight='bold',
             color='#1A1A1A', pad=10)
ax.legend(['One-time only', 'Returned to buy'],
          loc='lower center', fontsize=9, framealpha=0,
          bbox_to_anchor=(0.5, -0.06))

# Frequency distribution
ax2 = axes[1]
freq = cust_orders['order_count'].value_counts().sort_index().head(5)
bar_colors = ['#E8ECF0' if i == 1 else NAVY for i in freq.index]
bars = ax2.bar(freq.index, freq.values, color=bar_colors,
               width=0.6, edgecolor='white', zorder=3)
for bar, v in zip(bars, freq.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
             f'{v:,}', ha='center', va='bottom', fontsize=9, color='#1A1A1A')
ax2.set_xlabel('Number of orders placed', color=GRAY, fontsize=9.5)
ax2.set_ylabel('Number of customers', color=GRAY, fontsize=9.5)
ax2.set_title('Order Frequency Distribution', fontsize=11,
              fontweight='bold', color='#1A1A1A', pad=10)
ax2.spines['bottom'].set_color('#CCCCCC')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('reports/chart_repeat_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Delivery Performance vs Review Score

In [ ]:
def delivery_bucket(days):
    if pd.isna(days):  return None
    if days <= -3:     return '1. Early (3+ days)'
    if days <= 0:      return '2. On time'
    if days <= 3:      return '3. Late (1 to 3 days)'
    return '4. Late (3+ days)'

master['delivery_status'] = master['days_late'].apply(delivery_bucket)

del_review = (
    master
    .dropna(subset=['delivery_status', 'review_score'])
    .groupby('delivery_status')
    .agg(
        orders    = ('order_id', 'nunique'),
        avg_score = ('review_score', 'mean')
    )
    .reset_index()
)

print(del_review.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

labels = ['Early\n(3+ days)', 'On time', 'Late\n(1 to 3 days)', 'Late\n(3+ days)']
scores = del_review['avg_score'].values
counts = del_review['orders'].values
colors = [GREEN, BLUE, GOLD, RED]

bars = ax.bar(range(4), scores, color=colors, width=0.55,
              edgecolor='white', zorder=3)
ax.set_ylim(0, 5.5)
ax.set_ylabel('Average Review Score (out of 5)', color=GRAY, fontsize=10)
ax.set_xticks(range(4))
ax.set_xticklabels(labels, fontsize=10.5, color='#1A1A1A')
ax.axhline(y=4.0, color=GRAY, linestyle=':', linewidth=1.2, alpha=0.6)

for bar, score, cnt in zip(bars, scores, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.07,
            f'{score:.2f}', ha='center', fontsize=12,
            color='#1A1A1A', fontweight='bold')
    ax.text(bar.get_x() + bar.get_width()/2, 0.15,
            f'n={cnt:,}', ha='center', fontsize=8.5, color='white')

ax.set_title('How Delivery Timing Drives Customer Satisfaction',
             fontsize=12, fontweight='bold', color='#1A1A1A', pad=14, loc='left')
ax.spines['bottom'].set_color('#CCCCCC')

plt.tight_layout()
plt.savefig('reports/chart_delivery_vs_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

early_score = del_review[del_review['delivery_status'].str.contains('Early')]['avg_score'].values[0]
late_score  = del_review[del_review['delivery_status'].str.contains('3\+')]['avg_score'].values[0]
print(f'Score gap between early and very late: {early_score - late_score:.2f} points')

## 4. Seller Quality

In [ ]:
seller_perf = (
    master
    .groupby('seller_id')
    .agg(
        revenue       = ('order_value', 'sum'),
        orders        = ('order_id', 'nunique'),
        avg_score     = ('review_score', 'mean'),
        avg_days_late = ('days_late', 'mean')
    )
    .reset_index()
)

# Bottom 10% by review score
cutoff      = seller_perf['avg_score'].quantile(0.10)
bad_sellers = seller_perf[seller_perf['avg_score'] <= cutoff]
bad_rev     = bad_sellers['revenue'].sum()
total_rev   = seller_perf['revenue'].sum()

print(f'Total sellers               : {len(seller_perf):,}')
print(f'Bottom 10% score cutoff     : {cutoff:.2f}')
print(f'Sellers below threshold     : {len(bad_sellers):,}')
print(f'Revenue through bad sellers : R${bad_rev:,.0f}')
print(f'As % of total revenue       : {bad_rev/total_rev*100:.1f}%')

## 5. Geographic Analysis

In [ ]:
state_perf = (
    master
    .groupby('customer_state')
    .agg(
        revenue   = ('order_value', 'sum'),
        orders    = ('order_id', 'nunique'),
        avg_score = ('review_score', 'mean')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
    .head(10)
)

fig, ax1 = plt.subplots(figsize=(12, 4.5))

nat_avg   = master['review_score'].mean()
bar_colors = [RED if row['avg_score'] < nat_avg - 0.15 else NAVY
              for _, row in state_perf.iterrows()]

bars = ax1.bar(range(len(state_perf)), state_perf['revenue'],
               color=bar_colors, width=0.58, edgecolor='white', zorder=3)
ax1.set_ylabel('Total Revenue (R$)', color=GRAY, fontsize=10)
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M')
)
ax1.spines['bottom'].set_color('#CCCCCC')

ax2 = ax1.twinx()
ax2.plot(range(len(state_perf)), state_perf['avg_score'],
         color=GOLD, linewidth=2.2, marker='D', markersize=6,
         markerfacecolor='white', markeredgecolor=GOLD,
         markeredgewidth=1.8, zorder=4)
ax2.set_ylim(3.5, 4.5)
ax2.set_ylabel('Avg Review Score', color=GOLD, fontsize=10)
ax2.tick_params(axis='y', colors=GOLD)
ax2.spines['right'].set_color(GOLD)
ax2.spines['top'].set_visible(False)
ax2.axhline(y=nat_avg, color=GOLD, linestyle=':', alpha=0.4, linewidth=1)

ax1.set_xticks(range(len(state_perf)))
ax1.set_xticklabels(state_perf['customer_state'], fontsize=10.5)
ax1.set_title('Revenue by State vs Customer Satisfaction',
              fontsize=12, fontweight='bold', color='#1A1A1A', pad=14, loc='left')

red_p  = mpatches.Patch(color=RED,  label='Below avg satisfaction')
navy_p = mpatches.Patch(color=NAVY, label='Strong performers')
gold_l = plt.Line2D([0],[0], color=GOLD, linewidth=2, label='Review score')
ax1.legend(handles=[navy_p, red_p, gold_l], fontsize=9,
           loc='upper right', framealpha=0.8)

plt.tight_layout()
plt.savefig('reports/chart_geographic.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Payment Behaviour

In [ ]:
pay = (
    payments
    .merge(delivered[['order_id']], on='order_id', how='inner')
)

pay_summary = (
    pay
    .groupby('payment_type')
    .agg(
        orders    = ('order_id', 'nunique'),
        avg_value = ('payment_value', 'mean'),
        total_rev = ('payment_value', 'sum')
    )
    .reset_index()
    .sort_values('total_rev', ascending=False)
)

# Installment vs one-shot on credit card
cc = pay[pay['payment_type'] == 'credit_card'].copy()
cc['behaviour'] = cc['payment_installments'].apply(
    lambda x: 'Installment' if x > 1 else 'One-shot'
)

inst = (
    cc.groupby('behaviour')
    .agg(avg_value=('payment_value','mean'),
         count=('order_id','nunique'))
    .reset_index()
)

print(pay_summary.to_string(index=False))
print()
print(inst.to_string(index=False))

## 7. RFM Customer Segmentation

RFM scores every customer on three dimensions:

- **Recency** - how recently did they last buy? Score 5 = very recent, 1 = long time ago
- **Frequency** - how many times have they bought? Score 5 = very frequent
- **Monetary** - how much have they spent? Score 5 = highest spenders

Segments are then assigned based on combinations of these three scores.

In [ ]:
snapshot = master['order_purchase_timestamp'].max()

rfm = (
    master
    .groupby('customer_unique_id')
    .agg(
        recency   = ('order_purchase_timestamp', lambda x: (snapshot - x.max()).days),
        frequency = ('order_id', 'nunique'),
        monetary  = ('order_value', 'sum')
    )
    .reset_index()
)

# Score each dimension 1 to 5
# Recency: lower days = more recent = higher score
rfm['R'] = pd.qcut(rfm['recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'),
                    5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'],  5, labels=[1,2,3,4,5]).astype(int)

def assign_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    if r >= 3 and f >= 3:            return 'Loyal'
    if r >= 4 and f <= 2:            return 'New Customers'
    if r <= 2 and f >= 3:            return 'At Risk'
    if r <= 2 and f <= 2:            return 'Lost'
    return 'Potential'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

rfm_summary = (
    rfm.groupby('segment')
    .agg(
        customers     = ('customer_unique_id', 'count'),
        avg_revenue   = ('monetary', 'mean'),
        total_revenue = ('monetary', 'sum')
    )
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)

print(rfm_summary.to_string(index=False))

In [ ]:
seg_colors = {
    'Champions': GOLD, 'Loyal': GREEN, 'At Risk': RED,
    'New Customers': BLUE, 'Lost': GRAY, 'Potential': TEAL
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

segs   = rfm_summary['segment'].tolist()
colors = [seg_colors[s] for s in segs]

# Customers per segment
ax = axes[0]
bars = ax.barh(segs, rfm_summary['customers'], color=colors,
               edgecolor='white', height=0.6, zorder=3)
ax.set_xlabel('Number of customers', color=GRAY, fontsize=9.5)
ax.set_title('Customers per Segment', fontsize=11, fontweight='bold',
             color='#1A1A1A', pad=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, v in zip(bars, rfm_summary['customers']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{v:,}', va='center', fontsize=8.5)

# Revenue per segment
ax2 = axes[1]
bars2 = ax2.barh(segs, rfm_summary['total_revenue'], color=colors,
                 edgecolor='white', height=0.6, zorder=3)
ax2.set_xlabel('Total Revenue (R$)', color=GRAY, fontsize=9.5)
ax2.set_title('Revenue per Segment', fontsize=11, fontweight='bold',
              color='#1A1A1A', pad=10)
ax2.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M')
)
for bar, v in zip(bars2, rfm_summary['total_revenue']):
    ax2.text(bar.get_width() + 8000, bar.get_y() + bar.get_height()/2,
             f'R${v/1e6:.1f}M', va='center', fontsize=8.5)

plt.suptitle('RFM Customer Segmentation', fontsize=13,
             fontweight='bold', color='#1A1A1A', y=1.01)
plt.tight_layout()
plt.savefig('reports/chart_rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary of Findings

| Finding | Number | Implication |
|---------|--------|-------------|
| Customer repeat rate | 3.0% | Business runs on acquisition, no compounding |
| Score gap: early vs late delivery | 2.37 points | Late delivery is the primary driver of churn |
| At Risk segment revenue | R$3.7M | Highest priority for reactivation campaign |
| Revenue through bad sellers | R$476K | 3.1% of total revenue at risk from poor experience |
| RJ satisfaction vs SP | 3.87 vs 4.18 | Second biggest market has an operational problem |
| Installment vs one-shot spend | R$195 vs R$96 | Promoting installments lifts average order value |

---

### The One Recommendation

Fix the delivery experience. The 3% repeat rate is the number that 
determines everything else, and the delivery data shows exactly what 
is causing it. Late deliveries produce a 1.85 review score. Customers 
who give that score do not come back. Identifying and addressing the 
sellers and routes producing the most late deliveries is the single 
highest-leverage action available to this business.